In [14]:
from pathlib import Path
import pandas as pd

carpeta  = Path(r'source\trayectoria_por_sexo')

dfs = []

for archivo in sorted(carpeta.glob('*.csv')):
    df = pd.read_csv(archivo, sep=";",encoding='utf-8')

    df.columns = df.columns.str.strip().str.lower()

    df["anio"] = int(archivo.name[:4])

    dfs.append(df)

In [15]:
# Paso 3: comparar columnas entre archivos

# Armamos un diccionario con las columnas de cada archivo
columnas_por_archivo = {}
for f in sorted(carpeta.glob("*.csv")):
    df = pd.read_csv(f, sep=None, engine="python", encoding="utf-8", encoding_errors="replace")
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace("\ufeff", "",regex=False)
                  )
    columnas_por_archivo[f.name[:4]] = set(df.columns)

# Tomamos el primer año como referencia
anio_ref = list(columnas_por_archivo.keys())[0]
cols_ref = columnas_por_archivo[anio_ref]

print(f"Referencia: {anio_ref} → {sorted(cols_ref)}\n")

for anio, cols in columnas_por_archivo.items():
    if anio == anio_ref:
        continue
    faltantes = cols_ref - cols
    extras    = cols - cols_ref
    if not faltantes and not extras:
        print(f"✔  {anio}  → igual")
    else:
        print(f"✘  {anio}")
        if faltantes: print(f"     Faltan:  {sorted(faltantes)}")
        if extras:    print(f"     Sobran:  {sorted(extras)}")

Referencia: 2011 → ['ambito', 'departamento', 'entrados_1', 'entrados_10', 'entrados_11', 'entrados_12', 'entrados_1314', 'entrados_2', 'entrados_3', 'entrados_4', 'entrados_5', 'entrados_6', 'entrados_7', 'entrados_8', 'entrados_9', 'inicial_1', 'inicial_10', 'inicial_11', 'inicial_12', 'inicial_1314', 'inicial_2', 'inicial_3', 'inicial_4', 'inicial_5', 'inicial_6', 'inicial_7', 'inicial_8', 'inicial_9', 'm_entrados_1', 'm_entrados_10', 'm_entrados_11', 'm_entrados_12', 'm_entrados_1314', 'm_entrados_2', 'm_entrados_3', 'm_entrados_4', 'm_entrados_5', 'm_entrados_6', 'm_entrados_7', 'm_entrados_8', 'm_entrados_9', 'm_inicial_1', 'm_inicial_10', 'm_inicial_11', 'm_inicial_12', 'm_inicial_1314', 'm_inicial_2', 'm_inicial_3', 'm_inicial_4', 'm_inicial_5', 'm_inicial_6', 'm_inicial_7', 'm_inicial_8', 'm_inicial_9', 'm_nopromo_1', 'm_nopromo_10', 'm_nopromo_11', 'm_nopromo_12', 'm_nopromo_1314', 'm_nopromo_2', 'm_nopromo_3', 'm_nopromo_4', 'm_nopromo_5', 'm_nopromo_6', 'm_nopromo_7', 'm_no

In [ ]:
dfs = []
for f in sorted(carpeta.glob("*.csv")):
    
    # Paso 1: cargar
    df = pd.read_csv(f, sep=None, engine="python", encoding="utf-8")
    
    # Paso 2: normalizar + eliminar BOM
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace("\ufeff", "", regex=False))
    
    # Paso 3: extraer año del nombre del archivo
    df["anio"] = int(f.name[:4])
    
    # Paso 4: borrar columnas que terminan en _20
    cols_20 = [c for c in df.columns if c.endswith("_20")]
    df = df.drop(columns=cols_20)
    
    dfs.append(df)

# Paso 5: concatenar
df_final = pd.concat(dfs, ignore_index=True)

# Paso 6: anio primera columna
cols = ["anio"] + [c for c in df_final.columns if c != "anio"]
df_final = df_final[cols]

df_final.to_csv(r"data\trayectoria_por_sexo_final.csv", index=False, encoding="utf-8-sig")

print("Archivo exportado!")

print(df_final.shape)
print(df_final.head())

Archivo exportado!
(16702, 269)
   anio     provincia departamento   sector  ambito inicial_1 inicial_2  \
0  2011  Buenos Aires   25 DE MAYO  Estatal   Rural       123       165   
1  2011  Buenos Aires   25 DE MAYO  Estatal  Urbano       349       336   
2  2011  Buenos Aires   25 DE MAYO  Privado  Urbano       152       169   
3  2011  Buenos Aires   9 DE JULIO  Estatal   Rural       177       111   
4  2011  Buenos Aires   9 DE JULIO  Estatal  Urbano       568       543   

  inicial_3 inicial_4 inicial_5  ... m_otros_8 m_otros_9 m_otros_10  \
0       121       136       111  ...         0         0          0   
1       340       361       348  ...         0         0          0   
2       149       132       132  ...         0         0          0   
3       128       141       119  ...         0         0          0   
4       546       578       512  ...         4         0          0   

  m_otros_11 m_otros_12 m_otros_1314 primaria_egresados primaria_m_egresados  \
0         